<a href="https://colab.research.google.com/github/sampabiet90/AIML/blob/LogicMojo-AI-ML-April26-sampa90/oxford_iiit_pet_dataset_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:

from datasets import load_dataset
from copy import deepcopy
from pathlib import Path
import time

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms,models
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

In [3]:
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Device:", DEVICE)

Device: cuda


In [4]:
ds = load_dataset("timm/oxford-iiit-pet")

README.md:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  378MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  413MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3680 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3669 [00:00<?, ? examples/s]

In [5]:
train_ds= ds["train"]
label_names = train_ds.features["label"].names
print(len(label_names))

37


In [6]:
test_ds=ds["test"]

In [12]:
# take validation dataset from train_ds
split = train_ds.train_test_split(test_size=0.15,seed=42, stratify_by_column="label")
train_ds = split["train"]
val_ds = split["test"]

print(len(train_ds))
print(len(val_ds))
print(len(test_ds))

1387
245
3669


 Preprocessing and augmentation

In [24]:

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# large resolution to get confused breed more visible
IMG_SIZE = 256

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2,hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.08)),
])

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

Create Dataset Class

In [25]:
class PetDataset(Dataset):
  def __init__(self,ds,transform):
    self.ds=ds
    self.transform=transform

  def __len__(self):
    return len(self.ds)

  def __getitem__(self, index):
    ex= self.ds[index]
    image = ex['image'].convert("RGB")
    label = ex['label']
    if self.transform:
      image = self.transform(image)
    return image,label

Create DataLoader

In [26]:
train_loader = DataLoader(PetDataset(train_ds,train_transform),32,shuffle=True,num_workers=2)
val_loader = DataLoader(PetDataset(val_ds,transform),32,shuffle=False,num_workers=2)
test_loader = DataLoader(PetDataset(test_ds,transform),32,shuffle=False,num_workers=2)

Model Building

In [27]:

# chhoose resnet50 as it is more finegrained task
def build_resnet50(num_classes):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model  # nothing frozen -- every parameter trains, but at DIFFERENT rates.

model = build_resnet50(len(label_names))
model.to(DEVICE)

# discriminative learning rate in each layer as in this dataset it is required to find the breed ..more fine grained dataset

param_groups = [
    {"params": model.conv1.parameters(), "lr": 1e-6},
    {"params": model.bn1.parameters(), "lr": 1e-6},
    {"params": model.layer1.parameters(), "lr": 1e-6},
    {"params": model.layer2.parameters(), "lr": 1e-5},
    {"params": model.layer3.parameters(), "lr": 3e-5},
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
]

optimizer = optim.Adam(param_groups)
criterion = nn.CrossEntropyLoss()

Train

In [28]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    torch.set_grad_enabled(is_train)
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if is_train:
            optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        if is_train:
            loss.backward(); optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        n += images.size(0)
    torch.set_grad_enabled(True)
    return total_loss / n, correct / n


EPOCHS = 8
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
for epoch in range(1, EPOCHS + 1):
    tl, ta = run_epoch(model, train_loader, criterion, optimizer)
    vl, va = run_epoch(model, val_loader, criterion, None)
    history["train_loss"].append(tl); history["train_acc"].append(ta)
    history["val_loss"].append(vl); history["val_acc"].append(va)
    print(f"epoch {epoch}/{EPOCHS} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f}")

epoch 1/8 | train loss 2.2326 acc 0.4816 | val loss 0.4903 acc 0.8367
epoch 2/8 | train loss 0.4396 acc 0.8760 | val loss 0.2267 acc 0.9306
epoch 3/8 | train loss 0.2107 acc 0.9329 | val loss 0.1988 acc 0.9429
epoch 4/8 | train loss 0.1226 acc 0.9603 | val loss 0.1645 acc 0.9429
epoch 5/8 | train loss 0.1031 acc 0.9712 | val loss 0.1659 acc 0.9469
epoch 6/8 | train loss 0.0756 acc 0.9805 | val loss 0.1865 acc 0.9388
epoch 7/8 | train loss 0.0592 acc 0.9834 | val loss 0.1801 acc 0.9469
epoch 8/8 | train loss 0.0474 acc 0.9899 | val loss 0.2067 acc 0.9265


Evaluate

In [29]:

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        preds = model(images.to(DEVICE)).argmax(1).cpu()
        all_preds.append(preds); all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


test_preds, test_labels = collect_predictions(model, test_loader)
test_acc = (test_preds == test_labels).mean()
print(f"TEST accuracy (37 breeds): {test_acc:.4f}\n")
print(classification_report(test_labels, test_preds, target_names=label_names, digits=3))

TEST accuracy (37 breeds): 0.9022

                            precision    recall  f1-score   support

                abyssinian      0.852     0.939     0.893        98
          american_bulldog      0.576     0.990     0.728       100
 american_pit_bull_terrier      0.870     0.400     0.548       100
              basset_hound      0.989     0.890     0.937       100
                    beagle      0.882     0.970     0.924       100
                    bengal      0.793     0.920     0.852       100
                    birman      0.806     0.830     0.818       100
                    bombay      0.889     1.000     0.941        88
                     boxer      0.907     0.889     0.898        99
         british_shorthair      0.973     0.710     0.821       100
                 chihuahua      0.916     0.870     0.892       100
              egyptian_mau      0.903     0.866     0.884        97
    english_cocker_spaniel      0.979     0.950     0.964       100
            

In [30]:

# Instead of a dense, unreadable 37x37 heatmap, extract the top-15 most-confused
# (true, predicted) breed pairs directly from the confusion matrix.
cm = confusion_matrix(test_labels, test_preds)
np.fill_diagonal(cm, 0)  # zero out correct predictions -- we only want to rank ERRORS

pairs = []
for i in range(37):
    for j in range(37):
        if cm[i, j] > 0:
            pairs.append((cm[i, j], label_names[i], label_names[j]))
pairs.sort(reverse=True)

print(f"{'count':>6s}  {'true breed':30s}  {'predicted as':30s}")
for count, true_name, pred_name in pairs[:15]:
    print(f"{count:6d}  {true_name:30s}  {pred_name:30s}")

 count  true breed                      predicted as                  
    27  staffordshire_bull_terrier      american_bulldog              
    26  american_pit_bull_terrier       staffordshire_bull_terrier    
    24  american_pit_bull_terrier       american_bulldog              
    17  ragdoll                         birman                        
    14  british_shorthair               russian_blue                  
    13  birman                          siamese                       
    12  egyptian_mau                    bengal                        
    11  maine_coon                      ragdoll                       
    11  boxer                           american_bulldog              
    10  basset_hound                    beagle                        
     8  ragdoll                         siamese                       
     8  maine_coon                      bengal                        
     7  samoyed                         pomeranian                    
     6